In [1]:
# リカレントニューラルネットワーク（RNN）
# 確率と言語モデル
# word2vecを確率の視点から眺める
# 言語モデル
# CBOWモデルを言語モデルに？

In [ ]:
# RNNとは
# フィードフォワードとは
# 循環するニューラルネットワーク
# ループの展開
# Backpropagation Through Time
# Truncated BPTT
# Truncated BPTTのミニバッチ学習


In [2]:
# RNNとは
# 循環するニューラルネットワーク
# ループの展開
# Backpropagation Through Time
# Truncated BPTT
# Truncated BPTTのミニバッチ学習

In [ ]:
# RNNの実装
# RNNレイヤの実装
import numpy as np

class RNN:
    # Wx: 入力x用の重み  Wh: 前の隠れ状態h用の重み  b: バイアス
    def __init__(self, Wx, Wh, b):
        # 外部から受け取った重みをリストにまとめる
        # Optimizerで一気に更新できるようにするため
        # Optimizer = grads(勾配)を一気に更新する
        self.params = [Wx, Wh, b] 
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)] # 重みと全く同じ形をした「0」の箱
        self.cache = None  # forwardの最後でデータを一時保存するための箱

    def forward(self, x, h_prev):
        Wx, Wh, b = self.params
        # h_prev, h_next = 今何を覚えているかのメモ(記憶そのもの)
        # Wh = その気おkを次のステップでどう使いこなすかを示す重み
        # h_prevに重みを掛けることで、過去の情報（h_prev）をどう解釈するのかを学習する
        t = h_prev @ Wh + x @ Wx + b
        # なぜ「t」にnp.tanhを通すのか
        # tのままだと、足し算や掛け算を繰り返すうちに、数値がどんどん巨大になってしまう（勾配爆発）
        # 'np.tanh'を通すことで最大1.0、最小-1.0の間に保てる
        h_next = np.tanh(t)

        self.cache = (x, h_prev, h_next)  # backwardで使うためにメモを残しておく
        return h_next

    def backward(self, dh_next):
        Wx, Wh, b = self.params
        x, h_prev, h_next = self.cache  # forwardで保存したメモを取り出す

        dt = dh_next * (1 - h_next ** 2)
        db = np.sum(dt, axis=0)
        dWh = h_prev.T @ dt
        dh_prev = dt @ Wh.T
        dWx = x.T @ dt
        dx = dt @ Wx.T

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev

In [ ]:
# Embeddingレイヤの実装
class Embedding:
    def __init__(self, W):
        # paramsとgradsをリストにしておくのは、
        # Optimizerで一気に更新できるようにするため
        # Optimizer=gradsメモ（勾配）を一気に更新する
        self.params = [W]  
        self.grads = [np.zeros_like(W)]
        # 後でインデックスを受け取るための箱を準備しておく
        self.idx = None

    def forward(self, idx):
        W, = self.params  # 重みを取り出す
        self.idx = idx  # idxをself.idxとして受け取れるようにする
        out = W[idx]  # 重み行列から意味ベクトル（h）を抜き出す
        return out

    def backward(self, dout):
        dW, = self.grads  # 勾配を取り出す
        dW[...] = 0  # 前回の勾配を白紙にする

        for i, word_id in enumerate(self.idx):  # forwardで取り出した単語を順番に取り出す
            dW[word_id] += dout[i]  # その単語の勾配を足し合わせる

        return None